# MuseTalk 듀오 아바타 서버 (기술면접관 + 인성면접관 동시 등장, Colab GPU)

이 노트북은 **한 화면에 면접관 두 명이 같이 있는 영상**을 대상으로, 필요할 때(질문 사이 잡담, 또는 두 면접관이 동시에 말하는 상황)
각자의 얼굴만 립싱크해서 다시 합성하는 실험용 서버입니다. 기존 `musetalk_avatar_server.ipynb`(면접관별로 완전히 분리된 영상, 질문마다 한 명만 말하는 프로덕션 경로)는 그대로 두고, 이 기능은 별도 노트북에서 검증합니다.

## 핵심 아이디어: MuseTalk 내부는 건드리지 않는다
MuseTalk의 얼굴 검출/합성 로직은 "프레임에 얼굴 하나"를 전제로 만들어져 있습니다. 이걸 억지로 다중 얼굴 대응으로 뜯어고치는 대신:

1. **crop**: 듀오 영상을 ffmpeg로 좌/우(또는 실제 구도에 맞는 영역)로 나눠서, "기술면접관만 나온 영상" / "인성면접관만 나온 영상"을 각각 만듭니다. 이후 MuseTalk 입장에서는 기존과 완전히 동일한 "얼굴 하나짜리 영상"입니다.
2. **무음 패딩 오디오**: 각 아바타에게 전체 클립 길이만큼의 오디오 트랙을 하나씩 줍니다. 그 아바타가 말하는 구간에는 실제 음성을, 말하지 않는 구간에는 무음을 채웁니다. 이러면 "번갈아 말하기"와 "동시에 말하기"를 코드 분기 없이 같은 방식으로 처리할 수 있습니다 (각자 자기 트랙에서 소리가 나는 시점에만 입이 움직임).
3. **overlay**: 각 아바타의 립싱크 결과(크롭된 영상)를, 듀오 원본 프레임에서 원래 그 사람이 있던 자리에 다시 합성합니다. 두 아바타를 같은 배경 위에 동시에 겹쳐 그리므로, 동시에 말해도 문제없이 양쪽 다 입이 움직인 채로 한 화면에 나옵니다.

## 시작 전 체크리스트
1. **런타임 > 런타임 유형 변경**에서 GPU를 `A100` 또는 `L4`로 설정
2. Colab Secrets에 `HF_TOKEN`, `NGROK_AUTHTOKEN`, `NGROK_STATIC_DOMAIN` 등록 (기존 노트북과 동일). **주의**: 기존 프로덕션 서버 노트북과 동시에 켜두려면 ngrok 고정 도메인이 하나 더 필요합니다 (무료 플랜은 보통 계정당 1개 예약 도메인만 제공하므로, 추가로 예약하거나 유료 플랜을 확인하세요).
3. 듀오 영상을 Google Drive에 업로드: `MyDrive/musetalk_assets/avatar_duo.mp4` (두 면접관이 겹치지 않고 각자 프레임의 절반쯤을 차지하도록 촬영 — 카메라 고정, 사람도 크게 움직이지 않아야 crop 좌표가 전체 영상에서 안정적으로 맞습니다)
4. **cell "# 7. 듀오 영상 crop"에서 `CROP_REGIONS` 좌표를 실제 영상 구도에 맞게 확인/조정** — 기본값은 화면을 세로로 반씩 나누는 것으로 가정했습니다. 실제 영상에서 두 사람이 딱 절반씩 위치하지 않으면 조정이 필요합니다.

## 노트북 실행 순서
1~6번 셀: 환경 구축 (기존 노트북과 동일, Drive 캐시 있으면 빠르게 복원)
7번 셀: 듀오 영상 crop + 배경판(idle 프레임) 추출
8번 셀: 서버 스크립트 작성 (아바타 2개 준비 + 멀티 아바타 합성 로직)
9~10번 셀: 서버 기동
11~12번 셀: 로컬 루프백 벤치마크 (번갈아 말하기 / 동시에 말하기 두 케이스 모두 테스트)
13번 셀: ngrok 외부 노출


In [ ]:
# 1. GPU 확인 + Google Drive 마운트
!nvidia-smi --query-gpu=name,memory.total --format=csv

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Miniconda 설치 + Python 3.10 가상환경 생성 (Drive에 캐싱된 게 있으면 복원만 함)
# Colab 기본 런타임(3.12)과 별개로, MuseTalk 실행 전용 Python 3.10 환경을 만듭니다.
import os

CONDA_PREFIX = "/usr/local/miniconda"
ENV_NAME = "musetalk"
CONDA_BIN = f"{CONDA_PREFIX}/bin/conda"
CONDA_PY = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/python"
CONDA_PIP = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin/pip"

DRIVE_CONDA_CACHE = "/content/drive/MyDrive/musetalk_conda_cache/musetalk_conda_env.tar"

if not os.path.exists(CONDA_PY):
    if os.path.exists(DRIVE_CONDA_CACHE):
        print("[캐시 발견] Drive에서 conda 환경 전체를 복원합니다 (몇 분 소요)...")
        os.makedirs(CONDA_PREFIX, exist_ok=True)
        !tar -xf "{DRIVE_CONDA_CACHE}" -C {CONDA_PREFIX}
    else:
        print("[캐시 없음] Miniconda 최초 설치를 진행합니다...")
        if not os.path.exists(CONDA_BIN):
            !wget -O /content/miniconda.sh https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
            # -u: 대상 폴더가 이미 있어도(이전 시도 잔여물) 갱신 설치로 처리
            !bash /content/miniconda.sh -b -u -p {CONDA_PREFIX}
        # 최신 Anaconda 기본 채널(pkgs/main, pkgs/r)은 비대화형 설치 전에 이용약관 동의가 필요합니다 (최초 1회).
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
        !{CONDA_BIN} tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
        !{CONDA_BIN} create -y -n {ENV_NAME} python=3.10
else:
    print("conda 환경이 이미 준비되어 있습니다:", CONDA_PY)

# 셸 명령이 중간에 실패해도 파이썬 예외가 나지 않으므로, 실제로 준비됐는지 직접 확인합니다.
if os.path.exists(CONDA_PY):
    print("conda 환경 준비 완료:", CONDA_PY)
else:
    print("conda 환경 준비 실패. 아래 진단 정보를 확인하세요.")
    print("- conda 설치 여부:", os.path.exists(CONDA_BIN))
    !ls {CONDA_PREFIX}/envs 2>/dev/null || echo "envs 폴더 자체가 없음"
    raise RuntimeError("conda 환경이 생성되지 않았습니다. 위 로그를 보고 원인을 확인한 뒤 이 셀을 다시 실행하세요.")

# 노트북 커널(3.12) 쪽에서 필요한 도구 (ngrok 터널, 루프백 벤치마크용)
!pip install -q pyngrok websockets

In [ ]:
# 3. MuseTalk 레포 클론 (최초 1회)
# "폴더 존재 여부"가 아니라 "musetalk 소스 코드 존재 여부"로 판단합니다.
# (data/models 폴더는 체크포인트 다운로드 단계에서 os.makedirs로 미리 생겼을 수 있어,
# 폴더 존재만으로는 클론 완료 여부를 알 수 없습니다.)
%cd /content

if os.path.exists("/content/MuseTalk/musetalk"):
    print("MuseTalk 레포가 이미 준비되어 있습니다.")
elif not os.path.exists("/content/MuseTalk"):
    print("MuseTalk 폴더가 없어 새로 클론합니다...")
    !git clone https://github.com/TMElyralab/MuseTalk.git
else:
    print("MuseTalk 폴더는 있지만 레포 소스가 없습니다 (체크포인트만 먼저 받아진 상태).")
    print("기존 폴더(체크포인트 등)는 유지한 채 레포 소스만 받아옵니다...")
    %cd /content/MuseTalk
    !git init -q
    !git remote add origin https://github.com/TMElyralab/MuseTalk.git 2>/dev/null || git remote set-url origin https://github.com/TMElyralab/MuseTalk.git
    !git fetch --depth 1 origin main
    !git checkout -f main
    %cd /content

%cd /content/MuseTalk
print("musetalk 패키지 존재 여부:", os.path.exists("/content/MuseTalk/musetalk"))

In [ ]:
# 4. 검증된 버전 조합으로 conda 환경에 설치
# 아래 목록은 로컬(Windows, MoodTender 프로젝트)에서 실제로 동작을 확인한 조합입니다.
# tensorrt/pywin32/pyreadline3 등 Windows 전용 패키지는 Linux(Colab)와 무관하므로 제외했습니다.
# mmengine/mmcv/mmdet/mmpose는 mim으로 따로 설치합니다 (플랫폼별 prebuilt wheel을 정확히 찾아주기 때문).
# chumpy는 별도로 --no-build-isolation 설치합니다 (아래 참고).

requirements_text = """
numpy==1.23.5
basicsr==1.4.2
gfpgan==1.3.8
facexlib==0.3.0
diffusers==0.30.2
transformers==4.39.2
accelerate==0.28.0
huggingface-hub==0.30.2
tokenizers==0.15.2
safetensors==0.7.0
opencv-python==4.9.0.80
librosa==0.11.0
soundfile==0.12.1
ffmpeg-python==0.2.0
imageio==2.37.3
imageio-ffmpeg==0.6.0
moviepy==1.0.3
einops==0.8.1
omegaconf==2.3.0
scikit-learn==1.7.2
scipy==1.15.3
numba==0.65.1
tqdm==4.65.2
PyYAML==6.0.3
openai-whisper==20250625
fastapi==0.136.3
uvicorn==0.48.0
websockets==15.0.1
"""

with open("/content/musetalk_requirements.txt", "w") as f:
    f.write(requirements_text)

# torch는 cu118 전용 인덱스에서 받아야 해서 별도 설치
!{CONDA_PIP} install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu118 torch==2.0.1+cu118 torchvision==0.15.2+cu118 torchaudio==2.0.2+cu118

# numpy를 먼저 설치해둔 뒤, chumpy는 빌드 격리를 꺼서 이 numpy를 그대로 쓰게 합니다.
# (빌드 격리를 켠 채로 두면 chumpy가 격리된 새 환경에서 최신 numpy를 따로 받다가 깨집니다)
!{CONDA_PIP} install -q numpy==1.23.5
!{CONDA_PIP} install -q --no-build-isolation chumpy==0.70

!{CONDA_PIP} install -q -r /content/musetalk_requirements.txt

# OpenMMLab 계열은 mim으로 설치 (버전은 MoodTender에서 검증된 것과 동일)
!{CONDA_PIP} install -q --no-cache-dir -U openmim
!{CONDA_PY} -m mim install -q mmengine
!{CONDA_PY} -m mim install -q "mmcv==2.0.1"
!{CONDA_PY} -m mim install -q "mmdet==3.1.0"
!{CONDA_PY} -m mim install -q "mmpose==1.1.0"

# mim install 계열이 버전 고정 없이 opencv-python/numpy를 최신으로 끌어올리는 경우가 있어
# 마지막에 다시 원하는 버전으로 강제 고정합니다 (numpy 2.x가 들어오면 여러 패키지가 ABI 충돌로 깨짐).
!{CONDA_PIP} install -q "numpy==1.23.5" "opencv-python==4.9.0.80"

print("conda 환경 설치 완료 - 버전 확인:")
!{CONDA_PY} -c "import numpy, cv2, torch; print('numpy', numpy.__version__); print('opencv-python', cv2.__version__); print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())"

In [ ]:
# 4-1. conda 환경 전체를 Drive에 캐싱 (한 번만 하면 됨 - 다음 런타임부터는 2번 셀에서 자동 복원)
if not os.path.exists(DRIVE_CONDA_CACHE):
    print("conda 환경을 Drive에 캐싱합니다 (용량에 따라 몇 분 소요될 수 있습니다)...")
    !tar -cf /content/musetalk_conda_env.tar -C {CONDA_PREFIX} .
    os.makedirs(os.path.dirname(DRIVE_CONDA_CACHE), exist_ok=True)
    !cp /content/musetalk_conda_env.tar "{DRIVE_CONDA_CACHE}"
    print("Drive 캐싱 완료 - 다음부터는 런타임이 끊겨도 몇 분 안에 복구됩니다.")
else:
    print("conda 환경 캐시가 이미 Drive에 있습니다:", DRIVE_CONDA_CACHE)

In [ ]:
# 5. 체크포인트 다운로드 (Drive에 캐싱해서 다음 실행부터는 재다운로드 안 함)
# download_weights.sh는 구버전 huggingface-cli / gdown --id 문법에 의존하는데,
# 최신 huggingface_hub(1.x)에서 huggingface-cli가 폐지되고 gdown도 --id를 제거해서
# 일부 파일이 에러 없이 조용히 다운로드되지 않습니다. 이 셀은 그걸 감지해서 안정적인 방식으로 직접 받습니다.
# 또한 실패한 다운로드 시도가 "빈 파일"이나 "잘린 파일"을 남길 수 있어서, 존재 여부뿐 아니라
# 최소 크기까지 확인해서 손상된 파일을 걸러내고 다시 받습니다.

# HF_TOKEN은 노트북에 직접 적어두지 않고 Colab Secrets(왼쪽 사이드바 열쇠 아이콘)에서 읽어옵니다.
# "HF_TOKEN"이라는 이름으로 등록해두고, 이 노트북에 대한 접근 권한을 켜주세요.
from google.colab import userdata

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = ""

if HF_TOKEN and HF_TOKEN.startswith("hf_"):
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN 설정됨 - 속도 제한 없이 다운로드합니다.")
else:
    print("HF_TOKEN 미설정 - 로그인 없이 받아서 느릴 수 있습니다.")
    print("왼쪽 사이드바 열쇠(Secrets) 아이콘에서 'HF_TOKEN' 이름으로 등록해주세요. (선택사항이지만 권장)")

DRIVE_CKPT_DIR = "/content/drive/MyDrive/musetalk_checkpoints"
LOCAL_MODELS_DIR = "/content/MuseTalk/models"
DOWNLOAD_PATH_ENV = f"{CONDA_PREFIX}/envs/{ENV_NAME}/bin:" + os.environ["PATH"]

%cd /content/MuseTalk

# 레포를 갓 클론하면 models 폴더 자체가 없어서, 아래 cp 명령이 "target is not a directory"로
# 조용히 실패하고 캐시 복원 자체가 안 되는 문제가 있었습니다. 미리 만들어둡니다.
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)

# (경로, 최소 정상 크기 bytes) - 다운로드가 중간에 끊기면 파일은 있어도 크기가 비정상적으로 작습니다.
# sd-vae, whisper도 download_weights.sh 안에서 huggingface-cli로 받는 항목이라 같은 문제가 생길 수 있어 포함시킵니다.
REQUIRED_FILES = [
    ("models/dwpose/dw-ll_ucoco_384.pth", 100_000_000),
    ("models/musetalkV15/unet.pth", 10_000_000),
    ("models/musetalkV15/musetalk.json", 10),
    ("models/face-parse-bisent/79999_iter.pth", 10_000_000),
    ("models/sd-vae/config.json", 10),
    ("models/sd-vae/diffusion_pytorch_model.bin", 100_000_000),
    ("models/whisper/config.json", 10),
    ("models/whisper/pytorch_model.bin", 10_000_000),
    ("models/whisper/preprocessor_config.json", 10),
]

def checkpoints_complete():
    for rel_path, min_size in REQUIRED_FILES:
        full_path = f"/content/MuseTalk/{rel_path}"
        if not os.path.exists(full_path) or os.path.getsize(full_path) < min_size:
            return False
    return True

def remove_broken_files():
    for rel_path, min_size in REQUIRED_FILES:
        full_path = f"/content/MuseTalk/{rel_path}"
        if os.path.exists(full_path) and os.path.getsize(full_path) < min_size:
            print(f"손상된(너무 작은) 파일 삭제: {full_path} ({os.path.getsize(full_path)} bytes)")
            os.remove(full_path)

if os.path.exists(DRIVE_CKPT_DIR) and os.listdir(DRIVE_CKPT_DIR):
    print("[캐시 발견] Drive에서 체크포인트 복사 중...")
    !cp -r "$DRIVE_CKPT_DIR"/* "$LOCAL_MODELS_DIR"/

remove_broken_files()

if not checkpoints_complete():
    print("기본 다운로드 스크립트 실행 (일부 파일은 다음 단계에서 직접 보완합니다)...")
    !PATH="{DOWNLOAD_PATH_ENV}" sh ./download_weights.sh
    remove_broken_files()

if not checkpoints_complete():
    print("누락 파일을 직접 받습니다: 먼저 wget으로 huggingface.co resolve URL을 시도하고,")
    print("실패하면 huggingface_hub 파이썬 API로 재시도(최대 3회, Xet 가속 백엔드는 꺼서 표준 HTTP로 받고 이어받기 가능)합니다.")
    fix_script = '''
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"  # Xet 가속 백엔드가 간헐적으로 멈추는 이슈 회피, 표준 HTTP로 강제

import time, subprocess

hf_token = os.environ.get("HF_TOKEN")

targets = [
    ("yzd-v/DWPose", "dw-ll_ucoco_384.pth", "models/dwpose/dw-ll_ucoco_384.pth"),
    ("TMElyralab/MuseTalk", "musetalkV15/musetalk.json", "models/musetalkV15/musetalk.json"),
    ("TMElyralab/MuseTalk", "musetalkV15/unet.pth", "models/musetalkV15/unet.pth"),
    ("stabilityai/sd-vae-ft-mse", "config.json", "models/sd-vae/config.json"),
    ("stabilityai/sd-vae-ft-mse", "diffusion_pytorch_model.bin", "models/sd-vae/diffusion_pytorch_model.bin"),
    ("openai/whisper-tiny", "config.json", "models/whisper/config.json"),
    ("openai/whisper-tiny", "pytorch_model.bin", "models/whisper/pytorch_model.bin"),
    ("openai/whisper-tiny", "preprocessor_config.json", "models/whisper/preprocessor_config.json"),
]

def try_wget(repo_id, filename, dest):
    url = f"https://huggingface.co/{repo_id}/resolve/main/{filename}"
    cmd = ["wget", "-q", "-O", dest, url]
    if hf_token:
        cmd += ["--header", f"Authorization: Bearer {hf_token}"]
    result = subprocess.run(cmd)
    return result.returncode == 0 and os.path.exists(dest) and os.path.getsize(dest) > 0

def try_hf_hub_download(repo_id, filename, dest, attempts=3):
    from huggingface_hub import hf_hub_download
    import shutil
    for i in range(attempts):
        try:
            # force_download를 안 켜서, 중간에 끊겨도 다음 시도가 처음부터 다시 받지 않고 이어받습니다.
            path = hf_hub_download(repo_id=repo_id, filename=filename, token=hf_token)
            shutil.copy(path, dest)
            return True
        except Exception as e:
            print(f"  시도 {i+1}/{attempts} 실패: {e}")
            time.sleep(10)
    return False

for repo_id, filename, dest_rel in targets:
    dest = f"/content/MuseTalk/{dest_rel}"
    if os.path.exists(dest):
        continue
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    print(f"받는 중: {filename} ({repo_id})")
    if try_wget(repo_id, filename, dest):
        print("  wget으로 받음:", dest)
    elif try_hf_hub_download(repo_id, filename, dest):
        print("  huggingface_hub API로 받음:", dest)
    else:
        print("  받기 실패:", dest)
'''
    with open("/content/fix_downloads.py", "w") as f:
        f.write(fix_script)
    !{CONDA_PY} /content/fix_downloads.py
    remove_broken_files()

    face_parse_dest = "/content/MuseTalk/models/face-parse-bisent/79999_iter.pth"
    if not os.path.exists(face_parse_dest):
        os.makedirs(os.path.dirname(face_parse_dest), exist_ok=True)
        # gdown 최신 버전은 --id 옵션이 없어져서 파일 ID를 위치 인자로 바로 넘깁니다.
        !{CONDA_PY} -m gdown 154JgKpzCPW82qINcVieuPH3fZ2e0P812 -O {face_parse_dest}
    remove_broken_files()

# download_weights.sh / hf 관련 도구 설치 과정에서 huggingface_hub가 최신(1.x)으로 끌어올려지는 경우가 있습니다.
# transformers==4.39.2 / tokenizers==0.15.2는 huggingface_hub<1.0을 요구하므로, 다운로드가 끝난 뒤 다시 고정합니다.
!{CONDA_PIP} install -q "huggingface_hub==0.30.2"

if checkpoints_complete():
    os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
    !cp -r "$LOCAL_MODELS_DIR"/* "$DRIVE_CKPT_DIR"/
    print("모든 체크포인트 준비 완료 + Drive 백업 완료 (손상됐던 파일도 새로 백업되어 덮어써졌습니다)")
else:
    missing = [p for p, m in REQUIRED_FILES if not os.path.exists(f"/content/MuseTalk/{p}") or os.path.getsize(f"/content/MuseTalk/{p}") < m]
    print("여전히 누락/손상된 파일이 있습니다:", missing)

In [ ]:
# 7. 듀오 영상 준비: 25fps 변환 + 아바타별 crop + 배경판(idle 프레임) 추출
import subprocess, json as _json

DUO_SOURCE_PATH = "/content/drive/MyDrive/musetalk_assets/avatar_duo.mp4"
assert os.path.exists(DUO_SOURCE_PATH), f"듀오 영상을 찾을 수 없습니다: {DUO_SOURCE_PATH} - Drive 업로드 경로를 확인하세요."

os.makedirs("/content/MuseTalk/data", exist_ok=True)

DUO_SOURCE_VIDEO_25FPS = "/content/MuseTalk/data/avatar_duo_25fps.mp4"
!ffmpeg -y -i "$DUO_SOURCE_PATH" -r 25 "$DUO_SOURCE_VIDEO_25FPS"


def get_video_size(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height", "-of", "json", path],
        capture_output=True, text=True, check=True,
    )
    stream = _json.loads(result.stdout)["streams"][0]
    return stream["width"], stream["height"]


duo_width, duo_height = get_video_size(DUO_SOURCE_VIDEO_25FPS)
print(f"듀오 영상 해상도: {duo_width}x{duo_height}")

# TODO: 실제 영상에서 두 면접관이 차지하는 픽셀 영역에 맞게 조정하세요.
# 기본값: 화면을 세로로 정확히 반씩 나눔 (왼쪽=technical, 오른쪽=personality)
CROP_REGIONS = {
    "technical":   {"x": duo_width // 2, "y": 0, "w": duo_width - duo_width // 2, "h": duo_height},
    "personality": {"x": 0,              "y": 0, "w": duo_width // 2,            "h": duo_height},
}

CROPPED_VIDEOS = {}
for avatar_type, region in CROP_REGIONS.items():
    out_path = f"/content/MuseTalk/data/avatar_duo_{avatar_type}_crop.mp4"
    crop_filter = f"crop={region['w']}:{region['h']}:{region['x']}:{region['y']}"
    !ffmpeg -y -i "$DUO_SOURCE_VIDEO_25FPS" -vf "{crop_filter}" "$out_path"
    CROPPED_VIDEOS[avatar_type] = out_path

print("아바타별 crop 영상:", CROPPED_VIDEOS)

# 두 사람이 모두 가만히 있는(대사 없는) 순간의 프레임 하나를 전체 배경판으로 사용합니다.
# 오버레이 합성 시 이 이미지 위에 각 아바타의 립싱크 결과를 다시 얹습니다.
DUO_BACKGROUND_IMAGE = "/content/MuseTalk/data/avatar_duo_background.png"
!ffmpeg -y -i "$DUO_SOURCE_VIDEO_25FPS" -vframes 1 "$DUO_BACKGROUND_IMAGE"
print("배경판 이미지:", DUO_BACKGROUND_IMAGE)


In [ ]:
# 8. MuseTalk 듀오 서버 스크립트 작성
# 공용 모델은 한 번만 로드하고, 아바타 2개는 각자의 crop 영상으로 기존과 동일한 단일 얼굴 파이프라인을 씁니다.
# 새로 추가되는 부분은 "여러 아바타의 립싱크 결과를 무음 패딩 오디오 기반으로 만들고, 같은 배경 위에 동시에 합성"하는 로직입니다.

server_script = '\nimport os, sys, glob, time, types, shutil, subprocess, base64, uuid, json\nimport queue as _queue\nimport threading as _threading\nimport numpy as np\nimport cv2\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\n\nimport torch\nfrom fastapi import FastAPI, WebSocket, WebSocketDisconnect, Request\nfrom fastapi.responses import StreamingResponse\nimport uvicorn\nfrom transformers import WhisperModel\n\nfrom musetalk.utils.face_parsing import FaceParsing\nfrom musetalk.utils.utils import load_all_model, datagen\nfrom musetalk.utils.audio_processor import AudioProcessor\nfrom musetalk.utils.blending import get_image_blending\n\nimport scripts.realtime_inference as rt_module\n\nrt_module.args = types.SimpleNamespace(\n    version="v15",\n    ffmpeg_path="./ffmpeg-4.4-amd64-static/",\n    gpu_id=0,\n    vae_type="sd-vae",\n    unet_config="./models/musetalkV15/musetalk.json",\n    unet_model_path="./models/musetalkV15/unet.pth",\n    whisper_dir="./models/whisper",\n    inference_config="configs/inference/realtime.yaml",\n    bbox_shift=0,\n    result_dir="./results",\n    extra_margin=10,\n    fps=25,\n    audio_padding_length_left=2,\n    audio_padding_length_right=2,\n    batch_size=64,\n    output_vid_name=None,\n    use_saved_coord=False,\n    saved_coord=False,\n    parsing_mode="jaw",\n    left_cheek_width=90,\n    right_cheek_width=90,\n    skip_save_images=False,\n)\nargs = rt_module.args\n\ndevice = torch.device(f"cuda:{args.gpu_id}" if torch.cuda.is_available() else "cpu")\n\nprint("공용 모델 로딩 중 (VAE/UNet/Whisper/얼굴파싱)...", flush=True)\nvae, unet, pe = load_all_model(\n    unet_model_path=args.unet_model_path,\n    vae_type=args.vae_type,\n    unet_config=args.unet_config,\n    device=device,\n)\ntimesteps = torch.tensor([0], device=device)\npe = pe.half().to(device)\nvae.vae = vae.vae.half().to(device)\nunet.model = unet.model.half().to(device)\n\naudio_processor = AudioProcessor(feature_extractor_path=args.whisper_dir)\nweight_dtype = unet.model.dtype\nwhisper = WhisperModel.from_pretrained(args.whisper_dir)\nwhisper = whisper.to(device=device, dtype=weight_dtype).eval()\nwhisper.requires_grad_(False)\n\nfp = FaceParsing(left_cheek_width=args.left_cheek_width, right_cheek_width=args.right_cheek_width)\n\nprint("공용 모델 로드 완료", flush=True)\n\nrt_module.device = device\nrt_module.vae = vae\nrt_module.unet = unet\nrt_module.pe = pe\nrt_module.timesteps = timesteps\nrt_module.audio_processor = audio_processor\nrt_module.weight_dtype = weight_dtype\nrt_module.whisper = whisper\nrt_module.fp = fp\n\nAvatar = rt_module.Avatar\n\n# 듀오 영상에서 crop한 아바타별 영상 경로 (7번 셀에서 만든 것과 동일한 값을 그대로 사용)\nCROPPED_VIDEOS = {\n    "technical": "/content/MuseTalk/data/avatar_duo_technical_crop.mp4",\n    "personality": "/content/MuseTalk/data/avatar_duo_personality_crop.mp4",\n}\nDUO_BACKGROUND_IMAGE = "/content/MuseTalk/data/avatar_duo_background.png"\n# 스트리밍 합성 시 정지 사진 대신 이 영상을 반복 재생해서, 말하지 않는 아바타도\n# 자연스러운 미세 움직임(눈 깜빡임 등)을 유지하게 합니다 (7번 셀에서 만든 25fps 듀오 원본).\nDUO_IDLE_LOOP_VIDEO = "/content/MuseTalk/data/avatar_duo_25fps.mp4"\n\n_probe_result = subprocess.run(\n    ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", DUO_IDLE_LOOP_VIDEO],\n    capture_output=True, text=True, check=True,\n)\nDUO_IDLE_LOOP_DURATION = float(_probe_result.stdout.strip())\n\n# 문장별로 독립 호출되는 duo_stream_inference가 매번 처음(프레임 0)부터 다시 시작하지 않고,\n# 이전 문장이 끝난 지점부터 이어서 시작하도록 상태를 기억해둡니다.\n# (아바타별 얼굴 프레임 위치는 각자, 배경 루프 위치는 공용 하나로 관리)\n_avatar_frame_position = {}\n_background_position_seconds = [0.0]\n\n# crop 좌표(오버레이 합성 시 각 아바타를 원래 자리에 되돌려 놓기 위해 필요) - 7번 셀 CROP_REGIONS와 동일한 값이어야 함\nwith open("/content/MuseTalk/data/crop_regions.json") as f:\n    CROP_REGIONS = json.load(f)\n\n_bg_probe = cv2.imread(DUO_BACKGROUND_IMAGE)\nduo_height, duo_width = _bg_probe.shape[:2]\n\nAVATAR_TYPES = {\n    "technical": {"avatar_id": "duo_technical_interviewer", "source_video": CROPPED_VIDEOS["technical"]},\n    "personality": {"avatar_id": "duo_personality_interviewer", "source_video": CROPPED_VIDEOS["personality"]},\n}\n\ndef _smooth_coord_list(coord_list, window=3):\n    """\n    얼굴 bbox 좌표를 프레임 간 이동평균으로 살짝 스무딩합니다.\n    프레임마다 독립적으로 얼굴을 검출하다 보니 생기는 좌표 자체의 미세한 흔들림이\n    생성된 얼굴이 붙는 위치를 흔들어 떨림처럼 보이는 걸 줄이기 위함입니다.\n    """\n    n = len(coord_list)\n    half = window // 2\n    smoothed = list(coord_list)\n\n    for i in range(n):\n        if coord_list[i] is None:\n            continue\n        neighbors = [\n            coord_list[j]\n            for j in range(max(0, i - half), min(n, i + half + 1))\n            if coord_list[j] is not None\n        ]\n        if len(neighbors) >= 2:\n            avg = np.array(neighbors, dtype=np.float32).mean(axis=0)\n            smoothed[i] = tuple(int(round(v)) for v in avg)\n\n    return smoothed\n\n\ndef _smooth_mask_list(mask_list, window=3):\n    """\n    입 주변 블렌딩 마스크(픽셀 이미지)도 프레임 간 이동평균으로 살짝 스무딩합니다.\n    프레임마다 마스크 크기가 미세하게 다를 수 있어, 크기가 같은 이웃 프레임끼리만 섞습니다.\n    """\n    n = len(mask_list)\n    half = window // 2\n    smoothed = list(mask_list)\n\n    for i in range(n):\n        base = mask_list[i]\n        if base is None:\n            continue\n        neighbors = [\n            mask_list[j]\n            for j in range(max(0, i - half), min(n, i + half + 1))\n            if mask_list[j] is not None and mask_list[j].shape == base.shape\n        ]\n        if len(neighbors) >= 2:\n            avg = np.stack(neighbors).astype(np.float32).mean(axis=0)\n            smoothed[i] = avg.astype(base.dtype)\n\n    return smoothed\n\n\navatars = {}\noutput_dirs = {}\n\nfor avatar_type, cfg in AVATAR_TYPES.items():\n    avatar_id = cfg["avatar_id"]\n    drive_avatar_dir = f"/content/drive/MyDrive/musetalk_avatars/{avatar_id}"\n    local_avatar_dir = f"/content/MuseTalk/results/v15/avatars/{avatar_id}"\n\n    print(f"[{avatar_type}] 아바타 준비 시작 ({avatar_id})", flush=True)\n\n    if os.path.exists(drive_avatar_dir):\n        print(f"[{avatar_type}] [캐시 발견] Drive에서 복사", flush=True)\n        os.makedirs(os.path.dirname(local_avatar_dir), exist_ok=True)\n        os.system(f\'cp -r "{drive_avatar_dir}" "{local_avatar_dir}"\')\n        preparation = False\n    else:\n        print(f"[{avatar_type}] [캐시 없음] 최초 준비 (시간 소요)", flush=True)\n        preparation = True\n        if os.path.exists(local_avatar_dir):\n            shutil.rmtree(local_avatar_dir)\n\n    avatar = Avatar(\n        avatar_id=avatar_id,\n        video_path=cfg["source_video"],\n        bbox_shift=0,\n        batch_size=64,\n        preparation=preparation,\n    )\n    if preparation:\n        avatar.prepare_material()\n        os.makedirs(os.path.dirname(drive_avatar_dir), exist_ok=True)\n        os.system(f\'cp -r "{local_avatar_dir}" "{drive_avatar_dir}"\')\n        print(f"[{avatar_type}] 준비 결과 Drive 백업 완료", flush=True)\n\n    avatar.coord_list_cycle = _smooth_coord_list(avatar.coord_list_cycle)\n    # mask_coords_list_cycle은 mask_list_cycle(마스크 이미지)과 크기가 반드시 짝이 맞아야 해서\n    # 좌표만 따로 스무딩하면 안 됩니다 (스무딩된 좌표 크기 != 원본 마스크 이미지 크기로 어긋나서 블렌딩 오류 발생).\n    # mask_list_cycle은 원본과 같은 크기끼리만 섞도록 만들어져 있어 안전하게 그대로 스무딩합니다.\n    avatar.mask_list_cycle = _smooth_mask_list(avatar.mask_list_cycle)\n\n    avatars[avatar_type] = avatar\n    output_dirs[avatar_type] = f"{local_avatar_dir}/vid_output"\n\nprint("듀오 아바타 준비 완료:", list(avatars.keys()), flush=True)\n\n# 스트리밍은 처리량보다 첫 프레임이 얼마나 빨리 나오느냐가 중요해서,\n# 전체 생성용 batch_size(64)와 별도로 훨씬 작은 값을 씁니다.\nSTREAM_BATCH_SIZE = 16\n\n# A100/L4는 여유가 있어서, 속도용 프레임 보간(절반만 디코딩 후 평균내서 끼워넣기) 없이\n# 매 프레임을 다 디코딩합니다. 입 주변 미세한 떨림을 줄이는 대신 VAE 디코딩량이 늘어납니다.\n# 느려지면 이 값만 False로 되돌리면 됩니다.\nSTREAM_FULL_DECODE = True\n\n\ndef build_padded_track(turns_for_avatar, total_duration, out_path):\n    """\n    한 아바타가 가진 여러 발화 구간(turns_for_avatar: [{start, audio_path}, ...])을\n    total_duration짜리 무음 트랙 위에 각자의 시작 시각에 배치해서 하나의 오디오로 합칩니다.\n    발화가 겹치지 않는 게 정상이지만, 혹시 겹쳐도 amix로 자연스럽게 섞입니다.\n    """\n    inputs = ["-f", "lavfi", "-t", str(total_duration), "-i", "anullsrc=r=16000:cl=mono"]\n    filter_parts = []\n    mix_labels = ["[0:a]"]\n\n    for i, turn in enumerate(turns_for_avatar):\n        inputs += ["-i", turn["audio_path"]]\n        delay_ms = int(max(turn["start"], 0) * 1000)\n        label = f"[a{i}]"\n        filter_parts.append(f"[{i + 1}:a]adelay={delay_ms}|{delay_ms}[a{i}]")\n        mix_labels.append(label)\n\n    filter_parts.append(f"{\'\'.join(mix_labels)}amix=inputs={len(mix_labels)}:duration=first[mixed]")\n    filter_complex = ";".join(filter_parts)\n\n    cmd = ["ffmpeg", "-y", *inputs, "-filter_complex", filter_complex, "-map", "[mixed]", out_path]\n    subprocess.run(cmd, check=True, capture_output=True)\n\n\ndef run_avatar_inference(avatar_type, audio_path, out_name):\n    avatar = avatars[avatar_type]\n    output_dir = output_dirs[avatar_type]\n    avatar.inference(\n        audio_path=audio_path,\n        out_vid_name=out_name,\n        fps=25,\n        skip_save_images=False,\n    )\n    candidates = glob.glob(f"{output_dir}/{out_name}*.mp4")\n    if not candidates:\n        raise RuntimeError(f"[{avatar_type}] 출력 영상을 찾지 못했습니다: {output_dir}/{out_name}*")\n    return candidates[0]\n\n\ndef composite_duo_clip(avatar_clips, total_duration, out_path):\n    """\n    아바타별로 이미 립싱크된 전체 길이 클립(avatar_clips: {avatar_type: clip_path})을\n    듀오 배경판 위 각자의 원래 위치에 동시에 겹쳐서 하나의 영상으로 합칩니다.\n    """\n    inputs = ["-loop", "1", "-t", str(total_duration), "-i", DUO_BACKGROUND_IMAGE]\n    filter_parts = []\n    current_bg_label = "0:v"\n    audio_labels = []\n\n    for i, (avatar_type, clip_path) in enumerate(avatar_clips.items()):\n        inputs += ["-i", clip_path]\n        input_index = i + 1\n        region = CROP_REGIONS[avatar_type]\n        scale_label = f"[fg{i}]"\n        overlay_label = f"[bg{i}]"\n        filter_parts.append(\n            f"[{input_index}:v]scale={region[\'w\']}:{region[\'h\']}{scale_label}"\n        )\n        filter_parts.append(\n            f"[{current_bg_label}]{scale_label}overlay={region[\'x\']}:{region[\'y\']}{overlay_label}"\n        )\n        current_bg_label = overlay_label.strip("[]")\n        audio_labels.append(f"[{input_index}:a]")\n\n    filter_parts.append(f"{\'\'.join(audio_labels)}amix=inputs={len(audio_labels)}:duration=longest[aout]")\n    filter_complex = ";".join(filter_parts)\n\n    cmd = [\n        "ffmpeg", "-y", *inputs,\n        "-filter_complex", filter_complex,\n        "-map", f"[{current_bg_label}]", "-map", "[aout]",\n        "-c:v", "libx264", "-c:a", "aac", "-shortest",\n        out_path,\n    ]\n    subprocess.run(cmd, check=True, capture_output=True)\n\n\ndef synthesize_duo(turns, turn_id):\n    """\n    turns: [{"avatar_type": "technical", "start": 0.0, "audio_path": "..."}, ...]\n    같은 avatar_type이 여러 번 나올 수도 있고(같은 사람이 여러 문장), 서로 다른 avatar_type의\n    start 시각이 겹쳐도(동시에 말함) 상관없이 처리됩니다.\n    """\n    grouped = {}\n    for turn in turns:\n        grouped.setdefault(turn["avatar_type"], []).append(turn)\n\n    total_duration = 0.0\n    for turn in turns:\n        end = turn["start"] + turn.get("duration_hint", 4.0)\n        total_duration = max(total_duration, end)\n    total_duration += 0.5  # 약간의 여유\n\n    avatar_clips = {}\n    for avatar_type, avatar_turns in grouped.items():\n        padded_audio_path = f"/content/tmp_duo_padded_{avatar_type}_{turn_id}.wav"\n        build_padded_track(avatar_turns, total_duration, padded_audio_path)\n        clip_path = run_avatar_inference(avatar_type, padded_audio_path, f"duo_{avatar_type}_{turn_id}")\n        avatar_clips[avatar_type] = clip_path\n\n    final_path = f"/content/duo_final_{turn_id}.mp4"\n    composite_duo_clip(avatar_clips, total_duration, final_path)\n    return final_path\n\n\ndef _blend_fast(image, face, face_box, mask_array, crop_box):\n    """get_image_blending의 속도 최적화 버전. crop_box 영역만 numpy로 합성한다."""\n    x, y, x1, y1 = face_box\n    x_s, y_s, x_e, y_e = crop_box\n    h, w = image.shape[:2]\n    if x_s < 0 or y_s < 0 or x_e > w or y_e > h:\n        return get_image_blending(image, face, face_box, mask_array, crop_box)\n\n    region = image[y_s:y_e, x_s:x_e]\n    patch = region.copy()\n    patch[y - y_s:y1 - y_s, x - x_s:x1 - x_s] = face\n\n    mask = mask_array.astype(np.float32)\n    if mask.ndim == 2:\n        mask = mask[:, :, np.newaxis]\n    mask /= 255.0\n\n    region[:] = (patch.astype(np.float32) * mask + region.astype(np.float32) * (1.0 - mask)).astype(np.uint8)\n    return image\n\n\ndef _decode_fast(pred, full_decode=False):\n    """VAE 디코딩: 홀수 프레임만 디코딩하고 나머지는 보간해서 속도를 올립니다 (MoodTender stream_inference.py 이식)."""\n    latents = (1 / vae.scaling_factor) * pred\n    dtype = vae.vae.dtype\n    n = latents.shape[0]\n\n    if not full_decode and n >= 4:\n        key_latents = latents[::2]\n        key_decoded = vae.vae.decode(key_latents.to(dtype)).sample\n\n        k = key_decoded.shape[0]\n        if k >= 3:\n            prev_k = torch.cat([key_decoded[:1], key_decoded[:-1]])\n            next_k = torch.cat([key_decoded[1:], key_decoded[-1:]])\n            key_decoded = 0.15 * prev_k + 0.70 * key_decoded + 0.15 * next_k\n\n        interp = (key_decoded[:-1] + key_decoded[1:]) / 2\n        pairs = min(k - 1, interp.shape[0])\n        merged = torch.stack([key_decoded[:pairs], interp[:pairs]], dim=1)\n        merged = merged.reshape(pairs * 2, *key_decoded.shape[1:])\n        image = torch.cat([merged, key_decoded[pairs:], key_decoded[-1:]], dim=0)[:n]\n    else:\n        image = vae.vae.decode(latents.to(dtype)).sample\n\n        # 보간 없이 매 프레임을 그대로 디코딩하면 모델이 프레임마다 만들어내는\n        # 자연스러운 미세한 차이가 그대로 드러나 떨림처럼 보일 수 있습니다.\n        # 인접 프레임끼리 살짝만 섞어(0.15/0.70/0.15) 그 떨림을 줄입니다.\n        if image.shape[0] >= 3:\n            prev_img = torch.cat([image[:1], image[:-1]])\n            next_img = torch.cat([image[1:], image[-1:]])\n            image = 0.15 * prev_img + 0.70 * image + 0.15 * next_img\n\n    image = (image / 2 + 0.5).clamp(0, 1)\n    image = (image * 255).round().to(torch.uint8)\n    image = image[:, [2, 1, 0], :, :]\n    image = image.permute(0, 2, 3, 1)\n    return image.cpu().numpy()\n\n\ndef duo_stream_inference(avatar_type, audio_path, fps=25):\n    """\n    지정된 아바타(avatar_type) 한 명이 audio_path를 말하는 모습을,\n    듀오 배경(전체 프레임) 위 원래 자리에 실시간으로 겹쳐 합성하면서\n    fMP4 청크를 yield하는 제너레이터. (MoodTender의 stream_inference.py를\n    이식 + 듀오 배경 overlay를 같은 ffmpeg 파이프라인 안에 포함시킨 버전)\n    """\n    import time as _time\n    _t0 = _time.time()\n    print(f"[stream-timing][{avatar_type}] duo_stream_inference 시작", flush=True)\n\n    avatar = avatars[avatar_type]\n    region = CROP_REGIONS[avatar_type]\n\n    whisper_features, librosa_length = audio_processor.get_audio_feature(\n        audio_path, weight_dtype=weight_dtype\n    )\n    whisper_chunks = audio_processor.get_whisper_chunk(\n        whisper_features, device, weight_dtype, whisper, librosa_length,\n        fps=fps,\n        audio_padding_length_left=args.audio_padding_length_left,\n        audio_padding_length_right=args.audio_padding_length_right,\n    )\n    video_num = len(whisper_chunks)\n    print(f"[stream-timing][{avatar_type}] Whisper 특징 추출 완료: {_time.time() - _t0:.2f}초", flush=True)\n\n    # 이번 호출이 이어받을 시작 위치 (얼굴 프레임 인덱스 / 배경 루프 재생 위치)\n    start_frame_idx = _avatar_frame_position.get(avatar_type, 0)\n    start_bg_offset = _background_position_seconds[0] % DUO_IDLE_LOOP_DURATION\n\n    # 다음 호출이 이어받을 위치를 미리 갱신 (모듈로는 조회 시점에 적용되므로 그냥 누적)\n    _avatar_frame_position[avatar_type] = start_frame_idx + video_num\n    _background_position_seconds[0] += video_num / fps\n\n    H, W = avatar.frame_list_cycle[0].shape[:2]\n\n    cmd = [\n        "ffmpeg", "-y", "-v", "quiet", "-fflags", "+genpts",\n        "-f", "rawvideo", "-pix_fmt", "bgr24", "-s", f"{W}x{H}", "-r", str(fps),\n        "-thread_queue_size", "512", "-i", "pipe:0",\n        "-ss", f"{start_bg_offset:.3f}", "-stream_loop", "-1", "-i", DUO_IDLE_LOOP_VIDEO,\n        "-thread_queue_size", "512", "-i", audio_path,\n        "-filter_complex",\n        f"[1:v]scale={duo_width}:{duo_height}[bg];"\n        f"[bg][0:v]overlay={region[\'x\']}:{region[\'y\']}:shortest=1,format=yuv420p[outv]",\n        "-map", "[outv]", "-map", "2:a",\n        "-c:v", "libx264", "-preset", "ultrafast", "-tune", "zerolatency", "-crf", "23",\n        "-g", "12", "-sc_threshold", "0",\n        "-c:a", "aac", "-b:a", "128k",\n        "-movflags", "frag_keyframe+empty_moov+default_base_moof",\n        "-shortest", "-f", "mp4", "pipe:1",\n    ]\n    proc = subprocess.Popen(cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)\n\n    output_q = _queue.Queue()\n\n    _first_chunk_logged = [False]\n\n    def _read_stdout():\n        try:\n            while True:\n                chunk = proc.stdout.read(8192)\n                if not chunk:\n                    break\n                if not _first_chunk_logged[0]:\n                    print(f"[stream-timing][{avatar_type}] ffmpeg 첫 출력 청크: {_time.time() - _t0:.2f}초", flush=True)\n                    _first_chunk_logged[0] = True\n                output_q.put(chunk)\n        finally:\n            output_q.put(None)\n\n    reader = _threading.Thread(target=_read_stdout, daemon=True)\n    reader.start()\n\n    blend_q = _queue.Queue(maxsize=64)\n\n    def _gpu_loop():\n        gen = datagen(whisper_chunks, avatar.input_latent_list_cycle, STREAM_BATCH_SIZE)\n        batch_idx = 0\n        try:\n            for whisper_batch, latent_batch in gen:\n                batch_t0 = _time.time()\n                batch_n = whisper_batch.shape[0]\n\n                with torch.inference_mode():\n                    af = pe(whisper_batch.to(device))\n                    lb = latent_batch.to(dtype=unet.model.dtype)\n                    pred = unet.model(lb, timesteps, encoder_hidden_states=af).sample\n                unet_done = _time.time()\n\n                with torch.inference_mode():\n                    recon = _decode_fast(pred, full_decode=STREAM_FULL_DECODE)\n                vae_done = _time.time()\n\n                for raw in recon:\n                    blend_q.put(raw)\n\n                realtime_budget = batch_n / fps\n                total_batch_time = vae_done - batch_t0\n                behind = total_batch_time > realtime_budget\n                print(\n                    f"[batch-timing][{avatar_type}] 배치{batch_idx} n={batch_n} "\n                    f"UNet={unet_done - batch_t0:.3f}s VAE={vae_done - unet_done:.3f}s "\n                    f"합계={total_batch_time:.3f}s / 실시간예산={realtime_budget:.3f}s "\n                    f"{\'⚠️실시간보다느림\' if behind else \'OK\'}",\n                    flush=True,\n                )\n                batch_idx += 1\n        except Exception as e:\n            print(f"[duo-stream][{avatar_type}] GPU 루프 오류: {e}", flush=True)\n        finally:\n            blend_q.put(None)\n\n    def _cpu_blend_write():\n        idx = start_frame_idx\n        processed = 0\n        try:\n            while True:\n                raw = blend_q.get()\n                if raw is None or processed >= video_num:\n                    break\n                bbox = avatar.coord_list_cycle[idx % len(avatar.coord_list_cycle)]\n                ori = avatar.frame_list_cycle[idx % len(avatar.frame_list_cycle)].copy()\n                if bbox is None:\n                    proc.stdin.write(ori.tobytes())\n                    idx += 1\n                    processed += 1\n                    continue\n                x1, y1, x2, y2 = bbox\n                rf = cv2.resize(raw.astype(np.uint8), (x2 - x1, y2 - y1))\n                mask = avatar.mask_list_cycle[idx % len(avatar.mask_list_cycle)]\n                mc = avatar.mask_coords_list_cycle[idx % len(avatar.mask_coords_list_cycle)]\n                frame = _blend_fast(ori, rf, bbox, mask, mc)\n                proc.stdin.write(frame.tobytes())\n                idx += 1\n                processed += 1\n        except Exception as e:\n            print(f"[duo-stream][{avatar_type}] CPU 블렌딩 오류: {e}", flush=True)\n        finally:\n            try:\n                proc.stdin.close()\n            except Exception:\n                pass\n\n    gpu_thread = _threading.Thread(target=_gpu_loop, daemon=True)\n    writer = _threading.Thread(target=_cpu_blend_write, daemon=True)\n    gpu_thread.start()\n    writer.start()\n\n    while True:\n        chunk = output_q.get()\n        if chunk is None:\n            break\n        yield chunk\n\n    writer.join(timeout=5)\n    proc.wait(timeout=5)\n\n\ndef _warmup_batch_sizes(avatar_type, max_batch_size=STREAM_BATCH_SIZE):\n    """\n    cudnn.benchmark 모드에서는 배치 크기가 처음 등장할 때마다 최적 커널을 탐색하느라\n    그 순간만 느려집니다. 실제 요청 도중에 그 비용이 끼어들지 않도록,\n    서버 기동 시 1~max_batch_size 배치 크기를 미리 한 번씩 돌려 캐싱해둡니다.\n    """\n    avatar = avatars[avatar_type]\n    dummy_audio_path = f"/content/_warmup_silence_{avatar_type}.wav"\n    subprocess.run(\n        ["ffmpeg", "-y", "-f", "lavfi", "-i", "anullsrc=r=16000:cl=mono", "-t", "3", dummy_audio_path],\n        check=True, capture_output=True,\n    )\n\n    whisper_features, librosa_length = audio_processor.get_audio_feature(\n        dummy_audio_path, weight_dtype=weight_dtype\n    )\n    whisper_chunks = audio_processor.get_whisper_chunk(\n        whisper_features, device, weight_dtype, whisper, librosa_length,\n        fps=25,\n        audio_padding_length_left=args.audio_padding_length_left,\n        audio_padding_length_right=args.audio_padding_length_right,\n    )\n\n    for bs in range(1, max_batch_size + 1):\n        chunk = whisper_chunks[:bs]\n        if len(chunk) < bs:\n            break\n        gen = datagen(chunk, avatar.input_latent_list_cycle, bs)\n        for whisper_batch, latent_batch in gen:\n            with torch.inference_mode():\n                af = pe(whisper_batch.to(device))\n                lb = latent_batch.to(dtype=unet.model.dtype)\n                pred = unet.model(lb, timesteps, encoder_hidden_states=af).sample\n                _decode_fast(pred, full_decode=STREAM_FULL_DECODE)\n            break\n\n    os.remove(dummy_audio_path)\n    print(f"[{avatar_type}] 배치 크기 1~{max_batch_size} 워밍업 완료", flush=True)\n\n\nfor _avatar_type in avatars:\n    _warmup_batch_sizes(_avatar_type)\n\n\napp = FastAPI()\n\n\n@app.websocket("/synthesize/duo")\nasync def duo_handler(websocket: WebSocket):\n    await websocket.accept()\n    turn_counter = 0\n    try:\n        while True:\n            message = await websocket.receive_json()\n            turn_counter += 1\n\n            turns_meta = message.get("turns", [])\n            turns = []\n            for i, turn in enumerate(turns_meta):\n                audio_bytes = base64.b64decode(turn["audio_base64"])\n                audio_path = f"/content/tmp_duo_turn_{turn_counter}_{i}.wav"\n                with open(audio_path, "wb") as f:\n                    f.write(audio_bytes)\n                turns.append({\n                    "avatar_type": turn["avatar_type"],\n                    "start": turn.get("start", 0.0),\n                    "duration_hint": turn.get("duration_hint", 4.0),\n                    "audio_path": audio_path,\n                })\n\n            try:\n                t0 = time.time()\n                final_path = synthesize_duo(turns, turn_counter)\n                elapsed = time.time() - t0\n\n                with open(final_path, "rb") as f:\n                    video_bytes = f.read()\n\n                print(f"[duo][turn {turn_counter}] 합성 {elapsed:.2f}초, 영상 {len(video_bytes)/1024:.1f}KB", flush=True)\n                await websocket.send_bytes(video_bytes)\n            except Exception as e:\n                await websocket.send_json({"error": str(e)})\n    except WebSocketDisconnect:\n        print("[duo] 클라이언트 연결 종료", flush=True)\n\n\n@app.post("/synthesize/duo/stream")\nasync def duo_stream_handler(request: Request):\n    """\n    아바타 한 명이 오디오 하나를 말하는 실시간 스트리밍 전용 엔드포인트.\n    응답은 fMP4 청크가 연속으로 오는 HTTP 스트림입니다 (websocket이 아님).\n    """\n    body = await request.json()\n    avatar_type = body.get("avatar_type", "technical")\n    audio_base64 = body.get("audio_base64")\n\n    if avatar_type not in avatars:\n        return {"error": f"알 수 없는 avatar_type: {avatar_type}"}\n    if not audio_base64:\n        return {"error": "audio_base64가 필요합니다."}\n\n    audio_bytes = base64.b64decode(audio_base64)\n    audio_path = f"/content/tmp_duo_stream_{avatar_type}_{uuid.uuid4().hex}.wav"\n    with open(audio_path, "wb") as f:\n        f.write(audio_bytes)\n\n    def generate():\n        try:\n            for chunk in duo_stream_inference(avatar_type, audio_path):\n                yield chunk\n        finally:\n            if os.path.exists(audio_path):\n                os.remove(audio_path)\n\n    return StreamingResponse(\n        generate(),\n        media_type="video/mp4",\n        headers={"Cache-Control": "no-cache", "X-Content-Type-Options": "nosniff"},\n    )\n\n\nif __name__ == "__main__":\n    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")\n'


with open("/content/MuseTalk/musetalk_server.py", "w", encoding="utf-8") as f:
    f.write(server_script)

# 서버 프로세스가 crop 좌표를 알 수 있도록 별도 json으로도 저장 (7번 셀 CROP_REGIONS와 동일 값)
with open("/content/MuseTalk/data/crop_regions.json", "w", encoding="utf-8") as f:
    _json.dump(CROP_REGIONS, f)

print("musetalk_server.py 작성 완료 (듀오 아바타 동시/순차 발화 합성 서버)")


## 서버 기동

동시 발화(overlay + amix)와 순차 발화(무음 패딩)를 모두 지원하는 `/synthesize/duo` 엔드포인트가 뜹니다.
기존 노트북의 서버 기동 셀과 동일한 방식으로, 로그에 `Uvicorn running on`이 찍힐 때까지 기다립니다.

In [ ]:
# 9. 서버 프로세스 실행 + 기동 대기
import subprocess, time

if "server_proc" in globals() and server_proc.poll() is None:
    print("이전 서버 프로세스 종료 중...")
    server_proc.kill()
    server_proc.wait()

log_path = "/content/musetalk_server.log"
log_file = open(log_path, "w")

server_env = dict(os.environ)
server_env["MPLBACKEND"] = "Agg"

server_proc = subprocess.Popen(
    [CONDA_PY, "/content/MuseTalk/musetalk_server.py"],
    cwd="/content/MuseTalk",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=server_env,
)
print(f"서버 프로세스 시작 (PID {server_proc.pid})")

content = ""
ready = False
for _ in range(120):
    with open(log_path) as f:
        content = f.read()
    if "Uvicorn running on" in content:
        ready = True
        break
    if server_proc.poll() is not None:
        print("서버 프로세스가 예기치 않게 종료되었습니다. 아래 로그를 확인하세요.")
        break
    time.sleep(5)

print("서버 기동 완료!" if ready else "아직 기동되지 않았습니다 - 이 셀을 다시 실행해서 계속 대기하거나 로그를 확인하세요.")
print("----- 최근 로그 -----")
print(content[-3000:])


## 로컬 루프백 벤치마크

두 가지 케이스를 모두 테스트합니다.
1. **순차(번갈아 말하기)**: 기술면접관이 먼저, 인성면접관이 나중에 말함
2. **동시 발화**: 둘 다 처음부터 동시에 말함

두 케이스 모두 같은 `/synthesize/duo` 엔드포인트, 같은 프로토콜을 쓰고 `start` 값만 다릅니다.

In [ ]:
# 10. 로컬 루프백 벤치마크 (순차 발화 / 동시 발화 두 케이스)
!pip install -q edge-tts

import asyncio, websockets, time, base64, json

TEXT_TECHNICAL = "방금 답변 흥미롭게 들었습니다."
TEXT_PERSONALITY = "네, 저도 인상 깊었어요."


async def make_tts_wav(text, out_path):
    mp3_path = out_path.replace(".wav", ".mp3")
    !edge-tts --voice ko-KR-SunHiNeural --text "{text}" --write-media {mp3_path}
    !ffmpeg -y -i {mp3_path} {out_path}


async def run_case(case_name, turns_config):
    turns_payload = []
    for cfg in turns_config:
        wav_path = f"/content/bench_{case_name}_{cfg['avatar_type']}.wav"
        await make_tts_wav(cfg["text"], wav_path)
        with open(wav_path, "rb") as f:
            audio_b64 = base64.b64encode(f.read()).decode("utf-8")
        turns_payload.append({
            "avatar_type": cfg["avatar_type"],
            "start": cfg["start"],
            "duration_hint": cfg.get("duration_hint", 4.0),
            "audio_base64": audio_b64,
        })

    async with websockets.connect("ws://localhost:8000/synthesize/duo", max_size=None, ping_interval=None) as ws:
        t0 = time.time()
        await ws.send(json.dumps({"turns": turns_payload}))
        result = await ws.recv()
        elapsed = time.time() - t0

        if isinstance(result, str):
            print(f"[{case_name}] 서버가 에러를 반환했습니다 (왕복 {elapsed:.2f}초): {result}")
            return

        out_path = f"/content/benchmark_duo_{case_name}.mp4"
        with open(out_path, "wb") as f:
            f.write(result)
        print(f"[{case_name}] 왕복 시간: {elapsed:.2f}초, 영상 크기: {len(result)/1024:.1f}KB -> {out_path}")


await run_case("sequential", [
    {"avatar_type": "technical", "start": 0.0, "text": TEXT_TECHNICAL},
    {"avatar_type": "personality", "start": 3.0, "text": TEXT_PERSONALITY},
])

await run_case("simultaneous", [
    {"avatar_type": "technical", "start": 0.0, "text": TEXT_TECHNICAL},
    {"avatar_type": "personality", "start": 0.0, "text": TEXT_PERSONALITY},
])


In [ ]:
#로그 확인
!tail -n 60 /content/musetalk_server.log

In [ ]:
# 11. ngrok 터널 오픈 (기존 프로덕션 서버와 별도 도메인 필요)
from pyngrok import ngrok, conf
from google.colab import userdata

NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
# 프로덕션 서버(musetalk_avatar_server.ipynb)와 동시에 켜둘 계획이 없어서,
# 기존에 쓰던 고정 도메인(NGROK_STATIC_DOMAIN)을 그대로 재사용합니다.
NGROK_STATIC_DOMAIN = userdata.get("NGROK_STATIC_DOMAIN")

conf.get_default().auth_token = NGROK_AUTHTOKEN

public_tunnel = ngrok.connect(8000, domain=NGROK_STATIC_DOMAIN)
print("MuseTalk 듀오 서버 공개 URL:", public_tunnel.public_url)
print("WebSocket 엔드포인트:", public_tunnel.public_url.replace("https://", "wss://") + "/synthesize/duo")


## 백엔드 연동 프로토콜

`wss://.../synthesize/duo`로 접속해서 아래 형태의 **JSON 텍스트 메시지**를 보내면 됩니다 (기존 `/synthesize/technical`처럼 raw 오디오 bytes가 아니라 JSON입니다).

```json
{
  "turns": [
    {"avatar_type": "technical", "start": 0.0, "duration_hint": 4.0, "audio_base64": "..."},
    {"avatar_type": "personality", "start": 3.0, "duration_hint": 4.0, "audio_base64": "..."}
  ]
}
```

- `avatar_type`: `"technical"` 또는 `"personality"`
- `start`: 전체 클립 타임라인에서 이 오디오가 시작되는 시각(초). 같은 값이면 동시에 말하는 것으로 처리됩니다.
- `duration_hint`: 대략적인 발화 길이(초) — 서버가 전체 클립 길이를 추정하는 데 씁니다. 실제 오디오 길이보다 짧게 잡으면 뒷부분이 잘릴 수 있으니 여유 있게 주세요.
- `audio_base64`: 그 사람의 대사 음성(TTS 결과)을 base64로 인코딩한 값

서버는 완성된 mp4 bytes를 응답으로 돌려줍니다 (실패 시 `{"error": "..."}`  JSON 문자열).

## 아직 안 된 것 (백엔드 쪽에서 이어서 구현 필요)
- 대화 대사 자체를 만드는 LLM 호출 (질문 평가 후 ~ 다음 질문 전 사이에 넣을 짧은 잡담 생성)
- 면접관별로 다른 TTS 목소리 지정 (지금 backend/llm.py는 `onyx` 하나만 사용)
- 이 노트북의 `/synthesize/duo` 결과를 받아서 프론트로 전달하는 새 websocket 메시지 타입 + 프론트 재생 로직
- **crop 좌표 검증**: 실제 듀오 영상을 넣고 7번 셀을 돌려서, 두 사람이 잘리지 않고 깔끔하게 반씩 나뉘는지 반드시 확인 후 조정
